# Quantum Models in CLARYON

Demonstrates CLARYON's quantum models on a small Iris dataset.

**Two encoding strategies:**
- **Angle encoding** (recommended for tabular data): 1 qubit per feature, no L2 normalization
- **Amplitude encoding** (required for NIfTI, available for comparative evaluation): log2(features) qubits, L2-normalized

Uses binary Iris (4 features) for fast execution.

In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score
from claryon.io.base import TaskType

# Binary Iris: setosa vs non-setosa, 4 features
iris = load_iris()
X = iris.data
y = (iris.target > 0).astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 1. Angle-Encoded PQK SVM (Recommended)

Uses angle encoding: each feature rotates a dedicated qubit. No L2 normalization.
Z-scored features go in directly — the model handles encoding internally.

4 features → 4 qubits. `bandwidth=0.5` is optimal.

In [ ]:
from claryon.models.quantum.angle_pqk_svm import AngleProjectedKernelSVM

# Z-score the data (angle encoding benefits from normalization)
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler().fit(X_train)
X_train_z = scaler.transform(X_train)
X_test_z = scaler.transform(X_test)

model = AngleProjectedKernelSVM(bandwidth=0.5, seed=42)
model.fit(X_train_z, y_train, TaskType.BINARY)
probs = model.predict_proba(X_test_z)
preds = model.predict(X_test_z)
bacc = balanced_accuracy_score(y_test, preds)
print(f"angle_pqk_svm  BACC={bacc:.3f}  (4 qubits, angle encoding)")

## 2. Amplitude-Encoded Models (Comparative)

These models use amplitude encoding: features are padded to 2^n and L2-normalized.
4 features → 2 qubits. Z-score is NOT applied.

Available for comparative evaluation and for NIfTI imaging workflows.

In [ ]:
# Amplitude encode the raw (non-z-scored) data
from claryon.encoding.amplitude import amplitude_encode_matrix
X_train_q, enc_info = amplitude_encode_matrix(X_train)
X_test_q, _ = amplitude_encode_matrix(X_test, pad_len=enc_info.encoded_dim)
n_qubits = enc_info.n_qubits
print(f"Encoded: {X_train_q.shape[1]} dims, {n_qubits} qubits")

In [ ]:
from claryon.models.quantum.kernel_svm import QuantumKernelSVM
from claryon.models.quantum.qdc_hadamard import QuantumDistanceClassifierHadamard
from claryon.models.quantum.quantum_gp import QuantumGaussianProcess

# Small subsets for speed
X_tr = X_train_q[:20]
y_tr = y_train[:20]
X_te = X_test_q[:10]
y_te = y_test[:10]

amplitude_models = {
    "kernel_svm": QuantumKernelSVM(n_qubits=n_qubits),
    "qdc_hadamard": QuantumDistanceClassifierHadamard(n_qubits=n_qubits),
    "quantum_gp": QuantumGaussianProcess(n_qubits=n_qubits, noise=0.4),
}

for name, model in amplitude_models.items():
    model.fit(X_tr, y_tr, TaskType.BINARY)
    preds = model.predict(X_te)
    bacc = balanced_accuracy_score(y_te, preds)
    print(f"{name:20s} BACC={bacc:.3f}  ({n_qubits} qubits, amplitude encoding)")

In [ ]:
# QNN (slower — uses PyTorch training loop)
from claryon.models.quantum.qnn import QuantumNeuralNetwork

qnn = QuantumNeuralNetwork(n_qubits=n_qubits, n_layers=2, epochs=30, lr=0.01, batch_size=10)
qnn.fit(X_tr, y_tr, TaskType.BINARY)
preds = qnn.predict(X_te)
bacc = balanced_accuracy_score(y_te, preds)
print(f"{'qnn':20s} BACC={bacc:.3f}  ({n_qubits} qubits, amplitude encoding)")

## 3. Geometric Difference Score (GDQ)

The GDQ score (Huang et al., 2021) evaluates whether a fidelity quantum kernel
captures structure inaccessible to classical kernels.

**Note**: GDQ applies to amplitude-encoded fidelity kernels only.
It does not evaluate projected quantum kernels (`angle_pqk_svm`) or
training-based models (`qnn`, `qcnn_muw`).

In [ ]:
from claryon.evaluation.geometric_difference import geometric_difference_score

# Compute quantum fidelity kernel matrix from kernel_svm
model_ksvm = amplitude_models["kernel_svm"]
model_ksvm._init_pl()
K_Q = model_ksvm._kernel_matrix(X_tr, X_tr)

gdq = geometric_difference_score(K_Q, X_train=X_tr)
print(f"GDQ score: {gdq:.4f} (>1 suggests quantum advantage for fidelity kernels)")